# CS F425 Deep Learning Project
## QLoRA Fine-Tuning of Phi-2 for Structured Data Analysis

Fine-tunes `microsoft/phi-2` (4-bit QLoRA) to act as an AI agent over `sales_data.csv`.  
Given a natural language query, the model outputs:
```json
{"actions": ["filter_data(column='year', value=2022)", "aggregate_sum(column='revenue')"], "answer": 52345678.12}
```

**Runtime target**: ≤6 hours on Colab free-tier T4 GPU.

## Cell 1 — Install Packages

In [1]:
# Upgrade packages to versions compatible with Colab Python 3.12.
# trl>=0.9.0 is required for SFTConfig; transformers>=4.41.0 satisfies sentence-transformers.
!pip install -q -U "transformers>=4.41.0" "peft>=0.10.0" "trl>=0.9.0" "accelerate>=0.29.3" "datasets>=2.19.0" "bitsandbytes>=0.44.0" einops

# IMPORTANT: After this cell completes, restart the runtime:
# Runtime > Restart session  (then run all cells from Cell 2 onwards)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.6/137.6 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 85.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.1/199.1 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 245.2/245.2 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 297.6/297.6 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.0/542.0 kB 21.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 16.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 172.0/172.0 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 25.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 90.6 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 11.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Cell 2 — Imports & GPU Check

In [ ]:
import torch
import json
import re
import random
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel
from datasets import Dataset

# SFTConfig was introduced in trl 0.9.0; fall back gracefully for older installs
try:
    from trl import SFTTrainer, SFTConfig
    _USE_SFTCONFIG = True
except ImportError:
    from trl import SFTTrainer
    from transformers import TrainingArguments as SFTConfig
    _USE_SFTCONFIG = False

import trl, transformers as _tf
print(f"trl         : {trl.__version__}  (SFTConfig available: {_USE_SFTCONFIG})")
print(f"transformers: {_tf.__version__}")

assert torch.cuda.is_available(), "No GPU — enable GPU runtime: Runtime > Change runtime type > T4 GPU"
print(f"GPU : {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

ImportError: cannot import name 'SFTConfig' from 'trl' (/usr/local/lib/python3.12/dist-packages/trl/__init__.py)

## Cell 3 — Mount Google Drive

Saves adapter weights and checkpoints to Drive so they persist across Colab sessions.

**One-time setup**: Upload these files to `MyDrive/CS_F425_Project/`:
- `sales_data.csv`
- `agent_trajectories_2k.json`
- `tool_executor.py`
- `run_pipeline.py`

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import sys, os
DRIVE_DIR = "/content/drive/MyDrive/CS_F425_Project"
ADAPTER_DIR = f"{DRIVE_DIR}/phi2-agent-adapter"
CKPT_DIR = f"{DRIVE_DIR}/phi2-agent-qlora"

os.makedirs(ADAPTER_DIR, exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)

# Make tool_executor.py importable
sys.path.insert(0, DRIVE_DIR)

print(f"Drive dir: {DRIVE_DIR}")
print(f"Contents: {os.listdir(DRIVE_DIR)}")

## Cell 4 — Load Data

In [ ]:
df = pd.read_csv(f"{DRIVE_DIR}/sales_data.csv")
print(f"sales_data shape: {df.shape}")
print(df.dtypes)
print(df.head(3))

with open(f"{DRIVE_DIR}/agent_trajectories_2k.json") as f:
    trajectories = json.load(f)
print(f"\nTrajectories loaded: {len(trajectories)}")
print("Sample entry:", json.dumps(trajectories[0], indent=2))

## Cell 5 — Pre-compute Answers

The training data has `query` + `actions` but **no answers**.  
We execute each action sequence against `sales_data.csv` using `ToolExecutor` to obtain the gold answer.

We also extend `parse_agent_action` to handle `aggregate_mean`, `aggregate_count`, and string filter values that the original `run_pipeline.py` misses.

In [ ]:
from tool_executor import ToolExecutor

def parse_agent_action(action_str):
    """Extended parser: handles integer and string filter values,
    plus aggregate_mean and aggregate_count."""
    if action_str.startswith("filter_data"):
        # Try integer value first
        m = re.search(r"column='([^']+)', value=(\d+)", action_str)
        if m:
            return {"tool": "filter", "args": {"column": m.group(1), "op": "==", "value": int(m.group(2))}}
        # Try string value
        m = re.search(r"column='([^']+)', value='([^']+)'", action_str)
        if m:
            return {"tool": "filter", "args": {"column": m.group(1), "op": "==", "value": m.group(2)}}

    elif action_str.startswith("group_by"):
        m = re.search(r"column='([^']+)'", action_str)
        if m:
            return {"tool": "groupby", "args": {"column": m.group(1)}}

    elif action_str.startswith("aggregate_sum"):
        m = re.search(r"column='([^']+)'", action_str)
        if m:
            return {"tool": "aggregate", "args": {"column": m.group(1), "agg": "sum"}}

    elif action_str.startswith("aggregate_mean"):
        m = re.search(r"column='([^']+)'", action_str)
        if m:
            return {"tool": "aggregate", "args": {"column": m.group(1), "agg": "mean"}}

    elif action_str.startswith("aggregate_count"):
        m = re.search(r"column='([^']+)'", action_str)
        if m:
            return {"tool": "aggregate", "args": {"column": m.group(1), "agg": "count"}}

    elif action_str.startswith("sort_by"):
        m = re.search(r"column='([^']+)', order='([^']+)'", action_str)
        if m:
            ascending = m.group(2) != "desc"
            return {"tool": "sort", "args": {"column": m.group(1), "ascending": ascending}}

    elif action_str.startswith("top_k"):
        m = re.search(r"k=(\d+)", action_str)
        if m:
            return {"tool": "topk", "args": {"k": int(m.group(1))}}

    return None  # unrecognised action


def result_to_python(result_df):
    """Convert ToolExecutor result DataFrame to a JSON-serialisable Python value."""
    if result_df is None or (hasattr(result_df, 'empty') and result_df.empty):
        return None
    # Single-cell DataFrame → scalar
    if result_df.shape == (1, 1):
        val = result_df.iloc[0, 0]
        return round(float(val), 4) if isinstance(val, float) else int(val) if hasattr(val, '__int__') else val
    # Single-column, multiple rows → list of values
    if result_df.shape[1] == 1:
        return result_df.iloc[:, 0].tolist()
    # Multi-column → list of row dicts
    records = result_df.to_dict(orient='records')
    # Round floats for cleaner JSON
    cleaned = []
    for row in records:
        cleaned.append({k: (round(v, 4) if isinstance(v, float) else v) for k, v in row.items()})
    return cleaned


def compute_answer(actions, df):
    """Parse + execute a list of action strings; return Python-serialisable answer."""
    parsed = []
    for a in actions:
        try:
            p = parse_agent_action(a)
        except Exception:
            p = None
        if p is not None:
            parsed.append(p)
    if not parsed:
        return None
    try:
        result = ToolExecutor(df.copy()).execute(parsed)
        return result_to_python(result)
    except Exception as e:
        return None


# Build gold training examples
training_data = []
skipped = 0
for entry in trajectories:
    answer = compute_answer(entry["actions"], df)
    if answer is not None:
        training_data.append({
            "query": entry["query"],
            "actions": entry["actions"],
            "answer": answer
        })
    else:
        skipped += 1

print(f"Valid examples : {len(training_data)} / {len(trajectories)}")
print(f"Skipped        : {skipped}")
print("\nSample:")
print(json.dumps(training_data[0], indent=2))

## Cell 6 — Schema String & Prompt Template

The prompt follows the same `### Task / ### Schema / ### Question / ### Answer` pattern as Lab02.  
The schema string is fixed — it is identical at training and inference time.

In [ ]:
SCHEMA = """Table: sales_data
Columns: date (date), year (int), month (int), city (str), region (str),
         product (str), category (str), revenue (float), units_sold (int),
         cost (float), profit (float)

Available actions (use exactly this syntax):
  filter_data(column='col', value=val)
  group_by(column='col')
  aggregate_sum(column='col')
  aggregate_mean(column='col')
  aggregate_count(column='col')
  sort_by(column='col', order='asc'|'desc')
  top_k(k=N)"""

PROMPT_TEMPLATE = """### Task
Analyze the sales data and answer the query.
Output ONLY a JSON object with an \"actions\" list and an \"answer\" field. No other text.

### Schema
{schema}

### Question
{question}

### Answer
"""

def make_prompt(question: str) -> str:
    return PROMPT_TEMPLATE.format(schema=SCHEMA, question=question)

# EOS token is set in Cell 8 after tokenizer load
EOS = None

def format_example(row):
    """Format one training example as the full training string."""
    answer_json = json.dumps({"actions": row["actions"], "answer": row["answer"]})
    return {"text": make_prompt(row["query"]) + answer_json + EOS}

# Preview one formatted prompt
print(make_prompt(training_data[0]["query"]))

## Cell 7 — Split Dataset

In [ ]:
random.seed(42)
random.shuffle(training_data)

TRAIN_SIZE = min(1800, int(len(training_data) * 0.9))
VALID_SIZE = 200

raw_train = training_data[:TRAIN_SIZE]
raw_valid = training_data[TRAIN_SIZE : TRAIN_SIZE + VALID_SIZE]

print(f"Train : {len(raw_train)}")
print(f"Valid : {len(raw_valid)}")

## Cell 8 — Load Tokenizer

In [ ]:
MODEL_NAME = "microsoft/phi-2"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
EOS = tokenizer.eos_token  # "<|endoftext|>"

print(f"EOS token: {EOS!r}")

# Sanity-check token length on a sample
sample_text = make_prompt(raw_train[0]["query"]) + \
              json.dumps({"actions": raw_train[0]["actions"], "answer": raw_train[0]["answer"]}) + EOS
n_tokens = len(tokenizer(sample_text).input_ids)
print(f"Sample token length: {n_tokens} (target ≤512)")

## Cell 9 — Build HuggingFace Datasets

In [ ]:
def fmt(row):
    return format_example(row)  # EOS is now set

train_ds = Dataset.from_list(raw_train).map(
    fmt, remove_columns=["query", "actions", "answer"]
)
valid_ds = Dataset.from_list(raw_valid).map(
    fmt, remove_columns=["query", "actions", "answer"]
)

print(f"Train dataset : {len(train_ds)} examples")
print(f"Valid dataset : {len(valid_ds)} examples")
print("\nSample text (truncated):")
print(train_ds[0]["text"][:500])

## Cell 10 — Load Phi-2 in 4-bit (QLoRA)

Config mirrors Lab02 exactly: NF4 double-quant, bfloat16 compute dtype, gradient checkpointing.

In [ ]:
compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
print(f"Compute dtype: {compute_dtype}")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

!nvidia-smi --query-gpu=memory.used,memory.total --format=csv,noheader

## Cell 11 — Apply LoRA Adapters

In [ ]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "dense", "fc1", "fc2"],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
# Expected: ~0.84% trainable (~23M of 2.8B params)

## Cell 12 — Fine-Tune with SFTTrainer

Config mirrors Lab02: cosine LR, paged AdamW-8bit, effective batch 16, 2 epochs.  
Checkpoints saved to Drive so training can resume on Colab disconnect.

In [ ]:
# Build training args — handles both SFTConfig (trl>=0.9) and TrainingArguments fallback
_common_args = dict(
    output_dir=CKPT_DIR,
    seed=42,
    num_train_epochs=2,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    optim="paged_adamw_8bit",
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    logging_steps=50,
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    report_to="none",
)

if _USE_SFTCONFIG:
    # SFTConfig carries SFT-specific params; eval_strategy renamed in newer versions
    import inspect as _inspect
    _sft_sig = _inspect.signature(SFTConfig.__init__).parameters
    _eval_key = "eval_strategy" if "eval_strategy" in _sft_sig else "evaluation_strategy"
    training_args = SFTConfig(
        **_common_args,
        **{_eval_key: "epoch"},
        max_seq_length=512,
        dataset_text_field="text",
        packing=False,
    )
    trainer_extra = {}
else:
    # Older trl: SFT-specific args go directly to SFTTrainer, not TrainingArguments
    import inspect as _inspect
    _ta_sig = _inspect.signature(SFTConfig.__init__).parameters
    _eval_key = "eval_strategy" if "eval_strategy" in _ta_sig else "evaluation_strategy"
    training_args = SFTConfig(**_common_args, **{_eval_key: "epoch"})
    trainer_extra = {"max_seq_length": 512, "dataset_text_field": "text", "packing": False}

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=valid_ds,
    **trainer_extra,
)

print(f"Training on {len(train_ds)} examples, validating on {len(valid_ds)}")
print(f"Epochs: {training_args.num_train_epochs}  |  Effective batch: 16")
trainer.train()

## Cell 13 — Save Adapter Weights to Drive

In [ ]:
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

print(f"Adapter saved to: {ADAPTER_DIR}")
print("Files:")
for fname in sorted(os.listdir(ADAPTER_DIR)):
    size_mb = os.path.getsize(f"{ADAPTER_DIR}/{fname}") / 1e6
    print(f"  {fname:<40s}  {size_mb:.2f} MB")

## Cell 14 — Inference Function

Given a natural language question, generates the JSON output and parses it back to a Python dict.

In [ ]:
def generate_answer(question: str, max_new_tokens: int = 200) -> dict:
    """Run the fine-tuned model on a question and return parsed JSON dict."""
    prompt = make_prompt(question)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    new_tokens = out[0][inputs["input_ids"].shape[1]:]
    raw = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        # Fallback: extract first {...} block
        m = re.search(r'\{.*\}', raw, re.DOTALL)
        if m:
            try:
                return json.loads(m.group())
            except json.JSONDecodeError:
                pass
        return {"raw_output": raw}

## Cell 15 — Evaluation on Held-Out Queries

Runs the model on a set of test questions and cross-checks the model's answer against
the ground-truth answer computed directly by `ToolExecutor`.

In [ ]:
test_queries = [
    "What is the total revenue for 2022?",
    "Which city had the highest profit in 2021? Top 1",
    "What is the average units sold per month in 2023?",
    "List top 3 cities by revenue in 2022.",
    "What is the total profit for the Electronics category?",
]

for q in test_queries:
    result = generate_answer(q)
    actions = result.get("actions", [])
    model_answer = result.get("answer", "N/A")

    # Cross-check: execute the model's own actions against the data
    verified_answer = compute_answer(actions, df) if actions else None

    print(f"Q: {q}")
    print(f"  Actions          : {actions}")
    print(f"  Model answer     : {model_answer}")
    print(f"  Verified answer  : {verified_answer}")
    print(f"  Match            : {str(model_answer) == str(verified_answer)}")
    print()